In [ ]:
pip install sentence-transformers torch

In [ ]:
import ast
import time
import warnings
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

In [ ]:
PATH_ARTICLES    = "nyt_articles.csv"
PATH_MOSTPOPULAR = "nyt_mostpopular_daily.csv"
PATH_TOPSTORIES  = "nyt_topstories_daily.csv"
OUTPUT_PREFIX    = "nyt_similarity"
TOP_N            = 5

In [ ]:
MODELS = {
    "minilm":   ("sentence-transformers/all-MiniLM-L6-v2",  "none"),
    "mpnet":    ("sentence-transformers/all-mpnet-base-v2",  "none"),
    "gist":     ("avsolatorio/GIST-Embedding-v0",            "none"),
    "bge_base": ("BAAI/bge-base-en",                         "bge"),
    "e5_base":  ("intfloat/e5-base",                         "e5"),
    "bge_m3":   ("BAAI/bge-m3",                              "bge"),
}

In [ ]:
# 2. HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def parse_list_field(val):
    """
    Parse a stringified Python list (des_facet style in mostpopular/topstories)
    into a space-joined string.
    e.g. "['Politics', 'Trump, Donald J']" → "Politics Trump, Donald J"
    """
    if pd.isna(val) or str(val).strip() in ("[]", ""):
        return ""
    try:
        items = ast.literal_eval(str(val))
        return " ".join([str(i) for i in items])
    except Exception:
        return str(val)


def parse_keywords_string(val):
    """
    Parse a semicolon-separated keyword string (nyt_articles style)
    into a space-joined string.
    e.g. "Politics; Trump, Donald J; Ukraine" → "Politics Trump Donald J Ukraine"
    """
    if pd.isna(val) or str(val).strip() == "":
        return ""
    parts = [p.strip() for p in str(val).split(";") if p.strip()]
    return " ".join(parts)


def build_text(row, title_col, abstract_col, keyword_col, keyword_style):
    """
    Combine title/headline + abstract + keywords into one string.
    keyword_style: 'list'   → des_facet style (Python list string)
                   'string' → semicolon-separated string (nyt_articles style)
    """
    title    = str(row.get(title_col,    "") or "")
    abstract = str(row.get(abstract_col, "") or "")
    keywords = (parse_list_field(row.get(keyword_col, ""))
                if keyword_style == "list"
                else parse_keywords_string(row.get(keyword_col, "")))
    return f"{title} {abstract} {keywords}".strip()


def apply_prefix(texts, prefix_style):
    """
    Apply the appropriate text prefix for each model family.

    - BGE models work best with a retrieval instruction prefix
    - E5 models require "query: " prepended to every input
    - All other models use the raw text
    """
    if prefix_style == "bge":
        instruction = "Represent this sentence for searching relevant passages: "
        return [instruction + t for t in texts]
    elif prefix_style == "e5":
        return ["query: " + t for t in texts]
    else:
        return texts


def get_embeddings(model, texts, prefix_style, batch_size=64):
    """
    Encode texts into L2-normalized embeddings.
    Normalization means cosine similarity = simple dot product (faster).
    """
    texts_prepared = apply_prefix(texts, prefix_style)
    embeddings = model.encode(
        texts_prepared,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True   # L2 normalize → cosine sim = dot product
    )
    return embeddings

In [ ]:
print("=" * 60)
print("STEP 1 – Loading and preparing datasets")
print("=" * 60)

STEP 1 – Loading and preparing datasets


In [ ]:
# ── nyt_articles ──────────────────────────────────────────────────────────────
df_art = pd.read_csv(PATH_ARTICLES, low_memory=False)
df_art = df_art[df_art["document_type"].str.lower() == "article"].copy()
df_art = df_art.dropna(subset=["headline", "abstract"], how="all").copy()
df_art["text"] = df_art.apply(
    lambda r: build_text(r, "headline", "abstract", "keywords", "string"), axis=1
)
df_art = df_art[df_art["text"].str.strip() != ""].reset_index(drop=True)
print(f"nyt_articles:    {len(df_art):,} articles")

nyt_articles:    4,099 articles


In [ ]:
# ── nyt_mostpopular ───────────────────────────────────────────────────────────
df_pop = pd.read_csv(PATH_MOSTPOPULAR, low_memory=False)
df_pop = df_pop[df_pop["type"].str.lower() == "article"].copy()
df_pop = df_pop.dropna(subset=["title", "abstract"], how="all").copy()
df_pop["text"] = df_pop.apply(
    lambda r: build_text(r, "title", "abstract", "des_facet", "list"), axis=1
)
df_pop = df_pop[df_pop["text"].str.strip() != ""].reset_index(drop=True)
print(f"nyt_mostpopular: {len(df_pop):,} articles")

nyt_mostpopular: 181 articles


In [ ]:
# ── nyt_topstories ────────────────────────────────────────────────────────────
df_top = pd.read_csv(PATH_TOPSTORIES, low_memory=False)
df_top = df_top[df_top["item_type"].str.lower() == "article"].copy()
df_top = df_top.dropna(subset=["title", "abstract"], how="all").copy()
df_top["text"] = df_top.apply(
    lambda r: build_text(r, "title", "abstract", "des_facet", "list"), axis=1
)
df_top = df_top[df_top["text"].str.strip() != ""].reset_index(drop=True)
print(f"nyt_topstories:  {len(df_top):,} articles\n")

nyt_topstories:  4,361 articles



In [ ]:
def compare_datasets(df_a, df_b, name_a, name_b,
                     title_col_a, title_col_b,
                     emb_a, emb_b, model_key, top_n=5):
    """
    For every article in df_a, find the top_n most similar articles in df_b
    using dot product on L2-normalized embeddings (= cosine similarity).

    Returns:
      matches_df  — one row per (source article × matched article)
      mean_score  — average top-1 similarity score (overall alignment signal)
    """
    print(f"    Computing {name_a} → {name_b} …", end=" ", flush=True)
    t0 = time.time()

    # Dot product on normalized vectors = cosine similarity
    # Shape: (len_a, len_b)
    sim_matrix = np.dot(emb_a, emb_b.T)

    rows = []
    for i in range(len(df_a)):
        sims        = sim_matrix[i]
        top_indices = np.argsort(sims)[::-1][:top_n]
        source_title = df_a.iloc[i].get(title_col_a, "N/A")

        for rank, j in enumerate(top_indices, start=1):
            rows.append({
                "model":            model_key,
                "comparison":       f"{name_a} vs {name_b}",
                "source_dataset":   name_a,
                "source_title":     source_title,
                "matched_dataset":  name_b,
                "matched_title":    df_b.iloc[j].get(title_col_b, "N/A"),
                "rank":             rank,
                "similarity_score": round(float(sims[j]), 4),
            })

    matches_df = pd.DataFrame(rows)
    mean_score = round(float(sim_matrix.max(axis=1).mean()), 4)
    print(f"done ({time.time()-t0:.1f}s)  |  mean top-1 score: {mean_score:.4f}")
    return matches_df, mean_score


In [ ]:
COMPARISONS = [
    # (df_a,   df_b,   name_a,           name_b,            title_col_a, title_col_b)
    (df_art, df_pop, "nyt_articles",    "nyt_mostpopular",  "headline",  "title"),
    (df_art, df_top, "nyt_articles",    "nyt_topstories",   "headline",  "title"),
    (df_pop, df_top, "nyt_mostpopular", "nyt_topstories",   "title",     "title"),
]

In [ ]:
all_matches  = []
summary_rows = []

for model_key, (model_name, prefix_style) in MODELS.items():

    print("=" * 60)
    print(f"MODEL: {model_key.upper()}  |  {model_name}")
    print("=" * 60)

    # Load model
    print(f"  Loading model …")
    t_load = time.time()
    model  = SentenceTransformer(model_name)
    print(f"  Loaded in {time.time()-t_load:.1f}s\n")

    # Encode all three datasets once per model
    # (encoding once and reusing is more efficient than encoding per comparison)
    print("  Encoding datasets …")
    emb_art = get_embeddings(model, df_art["text"].tolist(), prefix_style)
    emb_pop = get_embeddings(model, df_pop["text"].tolist(), prefix_style)
    emb_top = get_embeddings(model, df_top["text"].tolist(), prefix_style)
    print()

    # Run all three comparisons
    print("  Running comparisons …")
    for df_a, df_b, name_a, name_b, tcol_a, tcol_b in COMPARISONS:

        # Select the correct embedding matrix for each dataset
        emb_map = {
            "nyt_articles":    emb_art,
            "nyt_mostpopular": emb_pop,
            "nyt_topstories":  emb_top,
        }
        matches_df, mean_score = compare_datasets(
            df_a, df_b, name_a, name_b, tcol_a, tcol_b,
            emb_map[name_a], emb_map[name_b],
            model_key, top_n=TOP_N
        )
        all_matches.append(matches_df)
        summary_rows.append({
            "model":            model_key,
            "model_name":       model_name,
            "comparison":       f"{name_a} vs {name_b}",
            "mean_top1_score":  mean_score,
            "n_source":         len(df_a),
            "n_target":         len(df_b),
        })

    # Free memory before loading next model
    del model
    print()

MODEL: MINILM  |  sentence-transformers/all-MiniLM-L6-v2
  Loading model …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

  Loaded in 50.7s

  Encoding datasets …


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/69 [00:00<?, ?it/s]


  Running comparisons …
    Computing nyt_articles → nyt_mostpopular … done (2.4s)  |  mean top-1 score: 0.6142
    Computing nyt_articles → nyt_topstories … done (3.3s)  |  mean top-1 score: 0.6664
    Computing nyt_mostpopular → nyt_topstories … done (0.1s)  |  mean top-1 score: 0.9347

